# Daily Challenge: Semantic Search & Reranking with Pinecone

Full pipeline using **Pinecone's serverless index** and **reranking API** (`bge-reranker-v2-m3`).

Pipeline:
1. Paste your Pinecone API key (no OAuth, no login page)
2. Rerank a small Apple corpus
3. Create a serverless medical-notes index
4. Load, embed, and upsert notes
5. Semantic search + Pinecone reranking
6. Side-by-side comparison

## Part 1 · Install required libraries

In [ ]:
!pip install -U "pinecone==6.0.1" pandas torch transformers

## Part 2 · Paste your Pinecone API key

Get your key from **app.pinecone.io → API Keys**.
The cell uses `getpass` so the key is hidden while you type/paste it.

In [ ]:
import os
import getpass

# Paste your Pinecone API key when prompted — it will be hidden
os.environ["PINECONE_API_KEY"] = getpass.getpass("Pinecone API Key: ")
print("API key set.")

## Part 3 · Instantiate the Pinecone client

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
print("Pinecone client ready.")

## Part 4 · Define query & documents

Mixed corpus: **Apple Inc.** (products) vs **apple** (fruit) — tests contextual disambiguation.

In [ ]:
query = "Tell me about Apple's products"

documents = [
    "An apple is a sweet, edible fruit produced by an apple tree.",
    "Apple Inc. makes the iPhone, iPad, MacBook, and Apple Watch.",
    "Apples come in many varieties such as Granny Smith, Fuji, and Gala.",
    "Apple's macOS and iOS operating systems power millions of devices worldwide.",
    "Eating an apple a day is often associated with good health habits.",
]

print(f"Query: {query}\n")
for i, doc in enumerate(documents):
    print(f"  [{i}] {doc}")

## Part 5 · Call the Pinecone reranker

In [ ]:
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

print(f"Returned {len(reranked.data)} reranked results.")

## Part 6 · Inspect reranked results

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"  {i+1}. Score: {m.score:.4f} | {m.document.text}")

show_reranked_results(query, reranked.data)

### Observation

The reranker correctly prioritises **Apple Inc.** documents over fruit documents,
showing contextual understanding of the query intent despite the shared keyword.

## Part 7 · Configure and create a serverless index

In [ ]:
import time
from pinecone import ServerlessSpec

cloud      = os.getenv('PINECONE_CLOUD',  'aws')
region     = os.getenv('PINECONE_REGION', 'us-east-1')
spec       = ServerlessSpec(cloud=cloud, region=region)
index_name = 'medical-notes-index'

# Delete existing index if present
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)
    print(f"Deleted existing index '{index_name}'.")

# dimension=384 matches all-MiniLM-L6-v2; metric=cosine for sentence embeddings
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)

print(f"Index '{index_name}' created (dim=384, metric=cosine).")

## Part 8 · Download & preview the medical notes dataset

In [ ]:
import requests
import tempfile
import pandas as pd

url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "sample_notes_data.jsonl")
    r = requests.get(url)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)
    df = pd.read_json(path, orient='records', lines=True)

# Show number of rows and columns
print("Data shape:", df.shape)
df.head()

## Part 9 · Upsert data into the index

In [ ]:
index = pc.Index(name=index_name)

# Push all rows from the DataFrame into Pinecone
index.upsert_from_dataframe(df)

print("Upsert complete.")

## Part 10 · Wait for the index to be ready

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: {vector_count}")
    # Wait until at least 1 vector is indexed
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

## Part 11 · Define the embedding function

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer  = AutoTokenizer.from_pretrained(model_name)
emb_model  = AutoModel.from_pretrained(model_name)
emb_model.eval()

def get_embedding(input_question: str):
    """
    Encode text → 384-dim vector.
    Averages token embeddings across the sequence length dimension (dim=1).
    """
    encoded_input = tokenizer(
        input_question, padding=True, truncation=True, return_tensors='pt'
    )
    with torch.no_grad():
        model_output = emb_model(**encoded_input)
    # Average across the sequence length dimension (dim=1)
    embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding

test_vec = get_embedding("patient has chest pain")
print(f"Embedding dimension: {test_vec.shape[0]}")

## Part 12 · Run a semantic search query

In [ ]:
question = "patient has chest pain"
query    = get_embedding(question).tolist()

# Retrieve top 5 most similar notes
results = index.query(vector=[query], top_k=5, include_metadata=True)

# Sort by score descending
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

## Part 13 · Display initial search results

In [ ]:
def show_results(question, matches):
    print(f"Question: '{question}'")
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. ID: {match['id']}")
        print(f"      Score   : {match['score']}")
        print(f"      Metadata: {match['metadata']}")
        print()

show_results(question, sorted_matches)

## Part 14 · Prepare documents for reranking

In [ ]:
# Concatenate all metadata key:value pairs into a single 'reranking_field'
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join(
            f"{key}: {value}" for key, value in match['metadata'].items()
        )
    }
    for match in results['matches']
]

print("Transformed documents (preview):")
for doc in transformed_documents:
    print(f"  ID {doc['id']}: {doc['reranking_field'][:100]}...")

## Part 15 · Rerank with Pinecone inference API

In [ ]:
refined_query = "patient experiencing acute chest pain and shortness of breath"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

print(f"{len(reranked_results.data)} reranked results returned.")

## Part 16 · Display reranked results

In [ ]:
def show_reranked_results(question, matches):
    print(f"Question: '{question}'")
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. ID             : {match.document.id}")
        print(f"      Score         : {match.score:.4f}")
        print(f"      Reranking Field: {match.document.reranking_field}")
        print()

show_reranked_results(refined_query, reranked_results.data)

## Part 17 · Side-by-side comparison

In [ ]:
print("=" * 65)
print(f"INITIAL QUERY : '{question}'")
print(f"REFINED QUERY : '{refined_query}'")
print("=" * 65)

print("\n--- Initial semantic search order ---")
for i, match in enumerate(sorted_matches, start=1):
    print(f"  {i}. ID={match['id']}  score={match['score']:.4f}")

print("\n--- After Pinecone bge-reranker-v2-m3 ---")
for i, match in enumerate(reranked_results.data, start=1):
    print(f"  {i}. ID={match.document.id}  score={match.score:.4f}")

print("\n" + "=" * 65)
print("Reranking improves result order by jointly attending to")
print("query and document tokens (cross-encoder), unlike the")
print("bi-encoder dot-product used in the initial search.")

## Bonus · Ask more medical questions

In [ ]:
bonus_questions = [
    ("broken bone treatment",          "patient requires orthopedic surgery for fracture"),
    ("diabetes management",            "insulin dosage adjustment for type 2 diabetes"),
    ("high blood pressure medication", "hypertension treatment plan with ACE inhibitors"),
]

for search_q, rerank_q in bonus_questions:
    q_vec = get_embedding(search_q).tolist()
    hits  = index.query(vector=[q_vec], top_k=5, include_metadata=True)

    docs_for_rerank = [
        {
            'id': m['id'],
            'reranking_field': '; '.join(f"{k}: {v}" for k, v in m['metadata'].items())
        }
        for m in hits['matches']
    ]

    reranked = pc.inference.rerank(
        model="bge-reranker-v2-m3",
        query=rerank_q,
        documents=docs_for_rerank,
        rank_fields=["reranking_field"],
        top_n=1,
        return_documents=True,
    )

    best = reranked.data[0]
    print(f"Q: {rerank_q}")
    print(f"   Best match ID={best.document.id}  score={best.score:.4f}")
    print(f"   Field: {best.document.reranking_field[:100]}")
    print()

## Clean up · Delete the index

In [ ]:
pc.delete_index(name=index_name)
print(f"Index '{index_name}' deleted.")